<a href="https://colab.research.google.com/github/SamyElnagar11/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamyElnagar11/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one page** (one pseudonymized content item) **in one month**, built by aggregating the daily facts for that page.

**Time window:** the month `2026-03`, a mid-panel month, so history exists before and after it. The final month (June 2026) stays sealed as a test month and is not used to develop anything.

**Lane:** CTR / Engagement Opportunity Scoring. **Tables:** `fact_content_daily_performance` joined to `dim_content` on `content_hash_id`.

**What I rank (proxy):** `ctr_gap` = median CTR of the page's position tier minus the page's own CTR. It is a proxy, not proof that a title is bad.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields |
|---|---|
| Feature (planned, up to 5) | impressions, average position, position tier, content age, word count (final list in section 3) |
| Label / proxy | `ctr_gap` (computed from the same month, so a starting point, not a future outcome) |
| Context | `client_hash_id` (for client-level grouping and holdout only), `content_hash_id` (join key) |
| Excluded | the final month (June 2026 / `_sample`): it is the natural outcome window, so developing on it would peek at the future. Also raw URLs, queries, titles and client names: they are not in the release and must never be published. Also any column derived from the label: it would leak the answer. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':   f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':   f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows


In [ ]:
M = "report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'"
q1 = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys
    FROM {TABLES['fact_daily']} WHERE {M}
""").df()
print(q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    n_rows  distinct_keys
0  9841378        9841378


In [ ]:
q2 = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS pages,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day
    FROM {TABLES['fact_daily']} WHERE {M}
""").df()
print(q2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    n_rows   pages  first_day   last_day
0  9841378  331437 2026-03-01 2026-03-31


In [ ]:
q3 = con.sql(f"""
    SELECT COUNT(*) AS all_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
           SUM(CASE WHEN gsc_impressions IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_impressions
    FROM {TABLES['fact_daily']} WHERE {M}
""").df()
print(q3)
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()[["column_name","column_type"]])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   all_rows  ga4_available_rows  rows_with_impressions
0   9841378            413966.0              9841378.0
                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid 

**Five features (all built from March 2026 search data only):**
1. `impressions`: knowable at the decision moment because it is the sum of March daily impressions, all observed before the review starts.
2. `avg_position`: knowable because it is the impression-weighted March position, observed before the review.
3. `days_seen`: knowable because it counts the March days the page appeared, observed before the review.
4. `position_volatility`: knowable because it is the standard deviation of March daily positions, observed before the review.
5. `position_tier`: knowable because it is a bucket of `avg_position`, so it uses no information beyond March.

**Proxy label:** `ctr_gap` = median CTR of the page's position tier minus the page's own CTR. Note: it uses `ctr`, so `ctr` is NOT a feature.

In [ ]:
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions)                          AS impressions,
           SUM(gsc_clicks)                               AS clicks,
           SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions) AS avg_position,
           COUNT(*)                                      AS days_seen,
           STDDEV(gsc_avg_position)                      AS position_volatility
    FROM {TABLES['fact_daily']}
    WHERE {M} AND gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
""").df()

feat["ctr"] = feat["clicks"] / feat["impressions"]
print(len(feat), "pages with >=100 impressions in March")
feat[["impressions", "clicks", "avg_position", "days_seen", "position_volatility", "ctr"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101441 pages with >=100 impressions in March


,impressions,clicks,avg_position,days_seen,position_volatility,ctr
0,6523.0,7.0,6.893301,31,2.442255,0.001073
1,453.0,0.0,3.214128,31,2.481312,0.000000
2,5630.0,6.0,6.535346,31,1.243547,0.001066
3,4944.0,13.0,7.435680,31,0.983020,0.002629
4,429.0,1.0,3.871795,31,4.341012,0.002331


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import spearmanr

feat["position_tier"] = pd.cut(feat["avg_position"], [0, 3, 10, 20, 50, np.inf], labels=False, include_lowest=True)
feat["ctr_gap"] = feat.groupby("position_tier")["ctr"].transform("median") - feat["ctr"]

cols = ["impressions", "avg_position", "days_seen", "position_volatility", "position_tier"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr, te = next(gss.split(feat, groups=feat["client_hash_id"]))

def score(cols):
    m = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    m.fit(feat.iloc[tr][cols], feat.iloc[tr]["ctr_gap"])
    return spearmanr(m.predict(feat.iloc[te][cols]), feat.iloc[te]["ctr_gap"])[0]

print("Honest score (Spearman, held-out clients):", round(score(cols), 3))

Honest score (Spearman, held-out clients): 0.175


**The trap (deliberate leakage):** I added a column `leak` copied from the label `ctr_gap` and re-scored. The score jumped from 0.173 to 1.0, which is a perfect score for the wrong reason: the model was handed the answer. I deleted the column and kept the honest number, 0.173 (Spearman, held-out clients). In this dataset, the five features carry only a weak signal for ctr_gap. This is a single split, so I treat it as directional.

In [ ]:
feat["leak"] = feat["ctr_gap"] * 1.0   # derived from the label on purpose
print("Leaky score:", round(score(cols + ["leak"]), 3), " <- looks perfect, but it is cheating")

feat = feat.drop(columns=["leak"])
print("Honest score again:", round(score(cols), 3))

Leaky score: 1.0  <- looks perfect, but it is cheating
Honest score again: 0.175


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation of my slice:** only about 4.2% of March rows (413,966 of 9,841,378) have GA4 data (`ga4_data_available IS TRUE`), so I built every feature from search data only and cannot say anything about engagement for most pages.

Other limits:
- The history is an unbalanced panel: clients start tracking on different dates.
- One month cannot show seasonality or a lasting decline.
- Low-impression pages are noisy, so I kept pages with at least 100 impressions.
- This is observational data. It cannot show that a title or meta change causes a CTR change.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.